Yep — next is **`03_gold_dim_product`**.

# 03 - Gold Layer - Product Dimension

Create a business-ready Product Dimension from the validated Silver product data.

**Source:** `end-to-end_pipeline.silver.products`
**Target:** `end-to-end_pipeline.gold.dim_product`

**Model Role:** Dimension Table
**Business Key:** `product_id`
**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide descriptive product attributes for analyzing sales by product, category, subcategory, brand, supplier, product status, and launch date.

---

## Cell 1 - Profile Silver Product Data

**Description:**
Confirm that the Silver Products table is ready to become a Gold dimension. This checks product-key uniqueness, row count, and availability of the main business attributes needed for reporting.

```sql
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER PRODUCTS FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_product_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_ids,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
        AS null_product_ids,

    COUNT(DISTINCT category)
        AS categories,

    COUNT(DISTINCT subcategory)
        AS subcategories,

    COUNT(DISTINCT brand)
        AS brands,

    COUNT(DISTINCT product_status)
        AS product_statuses,

    MIN(launch_date)
        AS earliest_launch_date,

    MAX(launch_date)
        AS latest_launch_date

FROM `end-to-end_pipeline`.silver.products;
```

---

## Cell 2 - Inspect Product Business Attributes

**Description:**
Review the main descriptive product attributes that will be exposed through the Gold dimension. This helps confirm that the dimension supports meaningful business analysis by category, subcategory, brand, and status.

```sql
%sql

-- ============================================================
-- CELL 2: INSPECT PRODUCT BUSINESS ATTRIBUTES
-- Purpose: Review product distribution across business groups
-- ============================================================

SELECT
    category,
    subcategory,
    product_status,
    COUNT(*) AS product_count

FROM `end-to-end_pipeline`.silver.products

GROUP BY
    category,
    subcategory,
    product_status

ORDER BY
    category,
    subcategory,
    product_status;
```

---

## Cell 3 - Transform Silver → Gold Product Dimension

**Description:**
Create the Gold Product Dimension at **one row per product**.

This step exposes the descriptive attributes needed for analysis. Silver-layer cleaning is not repeated.

`product_id` remains the business key that will later connect `dim_product` to `fact_sales`.

```sql
%sql

-- ============================================================
-- CELL 3: CREATE GOLD PRODUCT DIMENSION
-- Grain: One row per product
-- Business Key: product_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_product AS

SELECT
    product_id,
    product_name,
    category,
    subcategory,
    brand,
    supplier_id,
    launch_date,
    product_status

FROM `end-to-end_pipeline`.silver.products;
```

### Why are `list_price` and `standard_cost` not included here?

For this project, I would keep the dimension mainly **descriptive**.

The transaction-specific price and cost used for calculating revenue/profit already exist in `silver.sales_transactions`, and those measures will belong in `gold.fact_sales`.

That prevents the Product Dimension from becoming a mixture of descriptive attributes and transactional measures.

---

## Cell 4 - Validate Gold Product Dimension

**Description:**
Validate that the Product Dimension maintains one row per product, contains valid business attributes, and preserves the expected Silver-to-Gold row relationship.

```sql
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD PRODUCT DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT product_id)
            AS distinct_product_ids,

        COUNT(*) - COUNT(DISTINCT product_id)
            AS duplicate_product_ids,

        SUM(
            CASE
                WHEN product_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_ids,

        SUM(
            CASE
                WHEN product_name IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_names,

        SUM(
            CASE
                WHEN category IS NULL THEN 1
                ELSE 0
            END
        ) AS null_categories,

        SUM(
            CASE
                WHEN product_status IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_status

    FROM `end-to-end_pipeline`.gold.dim_product
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.products
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.distinct_product_ids = v.total_rows
            AND v.duplicate_product_ids = 0
            AND v.null_product_ids = 0
            AND v.null_product_names = 0
            AND v.null_categories = 0
            AND v.null_product_status = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;
```

If this returns **PASS**, `03_gold_dim_product` is done.

Next would be **`04_gold_dim_store`**.
Yep — next is **`03_gold_dim_product`**.

# 03 - Gold Layer - Product Dimension

Create a business-ready Product Dimension from the validated Silver product data.

**Source:** `end-to-end_pipeline.silver.products`
**Target:** `end-to-end_pipeline.gold.dim_product`

**Model Role:** Dimension Table
**Business Key:** `product_id`
**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide descriptive product attributes for analyzing sales by product, category, subcategory, brand, supplier, product status, and launch date.

---

## Cell 1 - Profile Silver Product Data

**Description:**
Confirm that the Silver Products table is ready to become a Gold dimension. This checks product-key uniqueness, row count, and availability of the main business attributes needed for reporting.

```sql
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER PRODUCTS FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_product_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_ids,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
        AS null_product_ids,

    COUNT(DISTINCT category)
        AS categories,

    COUNT(DISTINCT subcategory)
        AS subcategories,

    COUNT(DISTINCT brand)
        AS brands,

    COUNT(DISTINCT product_status)
        AS product_statuses,

    MIN(launch_date)
        AS earliest_launch_date,

    MAX(launch_date)
        AS latest_launch_date

FROM `end-to-end_pipeline`.silver.products;
```

---

## Cell 2 - Inspect Product Business Attributes

**Description:**
Review the main descriptive product attributes that will be exposed through the Gold dimension. This helps confirm that the dimension supports meaningful business analysis by category, subcategory, brand, and status.

```sql
%sql

-- ============================================================
-- CELL 2: INSPECT PRODUCT BUSINESS ATTRIBUTES
-- Purpose: Review product distribution across business groups
-- ============================================================

SELECT
    category,
    subcategory,
    product_status,
    COUNT(*) AS product_count

FROM `end-to-end_pipeline`.silver.products

GROUP BY
    category,
    subcategory,
    product_status

ORDER BY
    category,
    subcategory,
    product_status;
```

---

## Cell 3 - Transform Silver → Gold Product Dimension

**Description:**
Create the Gold Product Dimension at **one row per product**.

This step exposes the descriptive attributes needed for analysis. Silver-layer cleaning is not repeated.

`product_id` remains the business key that will later connect `dim_product` to `fact_sales`.

```sql
%sql

-- ============================================================
-- CELL 3: CREATE GOLD PRODUCT DIMENSION
-- Grain: One row per product
-- Business Key: product_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_product AS

SELECT
    product_id,
    product_name,
    category,
    subcategory,
    brand,
    supplier_id,
    launch_date,
    product_status

FROM `end-to-end_pipeline`.silver.products;
```

### Why are `list_price` and `standard_cost` not included here?

For this project, I would keep the dimension mainly **descriptive**.

The transaction-specific price and cost used for calculating revenue/profit already exist in `silver.sales_transactions`, and those measures will belong in `gold.fact_sales`.

That prevents the Product Dimension from becoming a mixture of descriptive attributes and transactional measures.

---

## Cell 4 - Validate Gold Product Dimension

**Description:**
Validate that the Product Dimension maintains one row per product, contains valid business attributes, and preserves the expected Silver-to-Gold row relationship.

```sql
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD PRODUCT DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT product_id)
            AS distinct_product_ids,

        COUNT(*) - COUNT(DISTINCT product_id)
            AS duplicate_product_ids,

        SUM(
            CASE
                WHEN product_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_ids,

        SUM(
            CASE
                WHEN product_name IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_names,

        SUM(
            CASE
                WHEN category IS NULL THEN 1
                ELSE 0
            END
        ) AS null_categories,

        SUM(
            CASE
                WHEN product_status IS NULL THEN 1
                ELSE 0
            END
        ) AS null_product_status

    FROM `end-to-end_pipeline`.gold.dim_product
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.products
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.distinct_product_ids = v.total_rows
            AND v.duplicate_product_ids = 0
            AND v.null_product_ids = 0
            AND v.null_product_names = 0
            AND v.null_categories = 0
            AND v.null_product_status = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;
```

If this returns **PASS**, `03_gold_dim_product` is done.

Next would be **`04_gold_dim_store`**.
